# LAB

In [1]:
import io
import time
import numpy as np
import pandas as pd
from PIL import Image
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [9]:
df = pd.read_parquet('./dataset/train.parquet')

def procesar_imagen(bytes_img):
    im = Image.open(io.BytesIO(bytes_img)).convert('RGB')
    im = im.resize((20, 20), Image.BILINEAR)
    return np.asarray(im, dtype=np.float32).ravel()

X = np.stack(df['image'].map(lambda d: procesar_imagen(d['bytes'])))
y = df['label'].values.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

mu = np.mean(X_train, axis=0)
sigma = np.std(X_train, axis=0)
sigma[sigma == 0] = 1

X_train = (X_train - mu) / sigma
X_test = (X_test - mu) / sigma

print(f"Entrenamiento: {X_train.shape} | Prueba: {X_test.shape}")

Entrenamiento: (80000, 1200) | Prueba: (20000, 1200)


In [20]:
X_t = torch.from_numpy(X_train).float().cuda()
Y_t = torch.from_numpy(y_train).long().cuda()

def softmax(x):
    return torch.exp(x) / torch.exp(x).sum(axis=-1, keepdims=True)

def evaluate(model, x):
    model.eval()
    y_pred = model(x)
    y_probas = softmax(y_pred)
    return torch.argmax(y_probas, axis=1)


D_in, H, D_out = 1200, 256, 200

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
).to("cuda")

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

epochs = 100
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # reinicio de gradientes
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # actualización de pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(model, torch.from_numpy(X_test).float().cuda())
print(accuracy_score(y_test, y_pred.cpu().numpy()))

Epoch 10/100 Loss 5.30570
Epoch 20/100 Loss 5.28390
Epoch 30/100 Loss 5.26434
Epoch 40/100 Loss 5.24561
Epoch 50/100 Loss 5.22714
Epoch 60/100 Loss 5.20882
Epoch 70/100 Loss 5.19079
Epoch 80/100 Loss 5.17326
Epoch 90/100 Loss 5.15637
Epoch 100/100 Loss 5.14023
0.03275


In [21]:
model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
).to("cuda")

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

epochs = 100
batch_size = 256
log_each = 2
l = []
model.train()
batches = len(X_t) // batch_size

for e in range(1, epochs+1):
    _l = []
    for b in range(batches):
        x_b = X_t[b*batch_size:(b+1)*batch_size]
        y_b = Y_t[b*batch_size:(b+1)*batch_size]

        y_pred = model(x_b)
        loss = criterion(y_pred, y_b)
        _l.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    l.append(np.mean(_l))
    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(model, torch.from_numpy(X_test).float().cuda())
print(f"Exactitud (Mini-batch Manual): {accuracy_score(y_test, y_pred.cpu().numpy()):.4f}")

Epoch 2/100 Loss 4.81942
Epoch 4/100 Loss 4.65871
Epoch 6/100 Loss 4.55453
Epoch 8/100 Loss 4.47290
Epoch 10/100 Loss 4.40381
Epoch 12/100 Loss 4.34293
Epoch 14/100 Loss 4.28796
Epoch 16/100 Loss 4.23754
Epoch 18/100 Loss 4.19079
Epoch 20/100 Loss 4.14712
Epoch 22/100 Loss 4.10607
Epoch 24/100 Loss 4.06732
Epoch 26/100 Loss 4.03058
Epoch 28/100 Loss 3.99566
Epoch 30/100 Loss 3.96239
Epoch 32/100 Loss 3.93065
Epoch 34/100 Loss 3.90028
Epoch 36/100 Loss 3.87118
Epoch 38/100 Loss 3.84326
Epoch 40/100 Loss 3.81642
Epoch 42/100 Loss 3.79057
Epoch 44/100 Loss 3.76564
Epoch 46/100 Loss 3.74158
Epoch 48/100 Loss 3.71833
Epoch 50/100 Loss 3.69583
Epoch 52/100 Loss 3.67406
Epoch 54/100 Loss 3.65297
Epoch 56/100 Loss 3.63251
Epoch 58/100 Loss 3.61264
Epoch 60/100 Loss 3.59333
Epoch 62/100 Loss 3.57456
Epoch 64/100 Loss 3.55629
Epoch 66/100 Loss 3.53849
Epoch 68/100 Loss 3.52115
Epoch 70/100 Loss 3.50425
Epoch 72/100 Loss 3.48776
Epoch 74/100 Loss 3.47166
Epoch 76/100 Loss 3.45593
Epoch 78/100 Los

In [22]:
class DatasetPersonalizado(torch.utils.data.Dataset):
    def __init__(self, X, Y):
        # Mantenemos los tensores en CPU para la carga en memoria y se envían dinámicamente
        self.X = torch.from_numpy(X).float()
        self.Y = torch.from_numpy(Y).long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, ix):
        return self.X[ix], self.Y[ix]

dataset = DatasetPersonalizado(X_train, y_train)

In [23]:
dataloader = torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=True)

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
).to("cuda")

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

epochs = 100
log_each = 2
l = []
model.train()

for e in range(1, epochs+1):
    _l = []
    for x_b, y_b in dataloader:
        x_b, y_b = x_b.cuda(), y_b.cuda()

        y_pred = model(x_b)
        loss = criterion(y_pred, y_b)
        _l.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    l.append(np.mean(_l))
    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(model, torch.from_numpy(X_test).float().cuda())
print(f"Exactitud (DataLoader): {accuracy_score(y_test, y_pred.cpu().numpy()):.4f}")

Epoch 2/100 Loss 4.82312
Epoch 4/100 Loss 4.66335
Epoch 6/100 Loss 4.55916
Epoch 8/100 Loss 4.47730
Epoch 10/100 Loss 4.40783
Epoch 12/100 Loss 4.34659
Epoch 14/100 Loss 4.29124
Epoch 16/100 Loss 4.24050
Epoch 18/100 Loss 4.19349
Epoch 20/100 Loss 4.14957
Epoch 22/100 Loss 4.10839
Epoch 24/100 Loss 4.06943
Epoch 26/100 Loss 4.03244
Epoch 28/100 Loss 3.99730
Epoch 30/100 Loss 3.96387
Epoch 32/100 Loss 3.93189
Epoch 34/100 Loss 3.90129
Epoch 36/100 Loss 3.87199
Epoch 38/100 Loss 3.84394
Epoch 40/100 Loss 3.81692
Epoch 42/100 Loss 3.79085
Epoch 44/100 Loss 3.76590
Epoch 46/100 Loss 3.74185
Epoch 48/100 Loss 3.71850
Epoch 50/100 Loss 3.69603
Epoch 52/100 Loss 3.67424
Epoch 54/100 Loss 3.65315
Epoch 56/100 Loss 3.63274
Epoch 58/100 Loss 3.61291
Epoch 60/100 Loss 3.59365
Epoch 62/100 Loss 3.57493
Epoch 64/100 Loss 3.55669
Epoch 66/100 Loss 3.53897
Epoch 68/100 Loss 3.52170
Epoch 70/100 Loss 3.50482
Epoch 72/100 Loss 3.48843
Epoch 74/100 Loss 3.47240
Epoch 76/100 Loss 3.45676
Epoch 78/100 Los

In [15]:
def collate_fn(batch):
    return torch.stack([x for x, y in batch]), torch.stack([y for x, y in batch])

dataloader = torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=True, collate_fn=collate_fn)

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
).to("cuda")

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

epochs = 100
log_each = 5
l = []
model.train()

for e in range(1, epochs+1):
    _l = []
    for x_b, y_b in dataloader:
        x_b, y_b = x_b.cuda(), y_b.cuda()

        y_pred = model(x_b)
        loss = criterion(y_pred, y_b)
        _l.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    l.append(np.mean(_l))
    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")
        PATH = f"./checkpoint_{e}.pt"
        torch.save(model.state_dict(), PATH)

Epoch 5/10 Loss 4.60400
Epoch 10/10 Loss 4.40488


In [16]:
y_pred = evaluate(model, torch.from_numpy(X_test).float().cuda())
print(f"Exactitud Final: {accuracy_score(y_test, y_pred.cpu().numpy()):.4f}")

Exactitud Final: 0.1028
